# 01 · First-gift cohorts and the second-gift label

**Workstream: data and label construction — Bakul.** Everything downstream depends on this notebook,
so it is written to be read aloud, not skimmed.

The project asks one question: *a donor gave to a classroom for the first time — will they give
again within a year, and should our stakeholder spend one of her scarce follow-up slots on them?*

Turning that sentence into a column is most of the work, and it is where the project can quietly
die. This notebook does four things:

1. Confirms the donor id actually links gifts across projects — Albert's blocking check.
2. Builds the first-gift cohorts and the label, with every judgement call stated.
3. Cuts the time-based split.
4. Runs the honest baseline so every later improvement has something to be measured against.

The logic lives in `src/`, not in these cells, because four other notebooks import it and a
definition that exists in two places will drift.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

import pandas as pd
import config, labels, baselines

sys.path.insert(0, str(REPO / "evals"))
import score

# Point this at the real ICPSR donations file once it is downloaded to the VM.
# Until then it runs on the synthetic fixture: `python evals/make_fixture.py data/raw/fixture_donations.csv`
DONATIONS = REPO / "data" / "raw" / "fixture_donations.csv"
USING_FIXTURE = "fixture" in DONATIONS.name

if USING_FIXTURE:
    print("RUNNING ON SYNTHETIC DATA. Every number below is meaningless — mechanics only.")
print(f"Source: {DONATIONS}")

RUNNING ON SYNTHETIC DATA. Every number below is meaningless — mechanics only.
Source: /Users/bakulbadwal/Documents/GitHub/gbus8496-project/data/raw/fixture_donations.csv


## 1 · The check that gates everything

Albert, approving the proposal:

> *"As soon as possible, confirm that the donor ID in the public-use file actually links gifts
> across projects, because the whole project depends on it."*

He is right, and it is worth being explicit about why. Released datasets often mint a fresh
identifier per record to protect privacy. If DonorsChoose did that here, then every donor would
appear exactly once, every donor would look like a one-time giver **by construction**, our label
would be all zeros, and there would be no project — not because donors do not come back, but
because the file cannot see them coming back.

The test is not "do ids repeat" but "do ids repeat **across different projects**". Repeats within a
single project could just be an id scoped to that project.

In [2]:
donations = labels.load_donations(DONATIONS)
print(f"\n{len(donations):,} donation rows, {donations['donor_id'].nunique():,} distinct donors")

projects_per_donor = donations.groupby("donor_id")["project_id"].nunique()
multi = (projects_per_donor > 1).mean()
print(f"Donors giving to more than one distinct project: {multi:.1%}")
print("PASS — ids follow the person." if multi > 0.05 else "STOP — ids do not link. Switch to the fallback.")

Resolved columns:
  donor_id    → DONOR_ID
  month       → CREATED_MONTH
  amount      → AMOUNT
  project_id  → PROJECTID
  chunk 1: 71,292 rows

71,292 donation rows, 40,000 distinct donors
Donors giving to more than one distinct project: 50.1%
PASS — ids follow the person.


## 2 · Building the label, and the three calls it rests on

`labels.build_cohorts()` reduces the donation rows to one row per donor. Three decisions are baked
in, and each could defensibly have gone the other way — so each is stated here rather than buried.

**(a) Same-month repeats do not count as a second gift.**
DonorsChoose lets a donor fund several classrooms in a single checkout, and this release records
month, not day. So two donations in the donor's first month are more plausibly *one* giving event
split across classrooms than a genuine return visit. Counting them would inflate the positive class
with something our stakeholder cannot act on — she cannot steward back a donor who never left. The
diagnostics below report the rate both ways so the choice is visible, not assumed.

**(b) The window is twelve whole calendar months, M+1 to M+12.** No day precision exists, so no
finer definition is available.

**(c) Cohorts whose window has not closed are dropped, not labelled zero.**
A donor whose first gift is in mid-2019 has not had a year to come back — the data simply stops.
Labelling them 0 would teach the model that recent donors do not return, which is an artefact of
the cutoff rather than a fact about donors.

In [3]:
cohorts, diagnostics = labels.build_cohorts(donations)
cohorts.head()


Label construction:
  donors_before_window_filter              40,000
  positive_rate_strict                     0.3056
  share_with_multiple_first_month_gifts    0.2180
  donors_dropped_unclosed_window           1,236
  donors_dropped_pre_coverage              0
  donors_labelled                          38,764
  positive_rate                            0.3022

By split:
         donors  positive_rate
split                         
holdout    4865       0.373690
train     33899       0.291985


,donor_id,cohort_month,first_gift_amount,first_month_gifts,first_project_id,second_gift_amount,second_gift_count,gave_again,cohort,split
0,d0000000,95,17.34,1,p297220,0.00,0,0,2007-12,train
1,d0000001,150,35.49,1,p045416,0.00,0,0,2012-07,train
3,d0000003,209,10.01,1,p096848,0.00,0,0,2017-06,holdout
4,d0000004,173,104.40,1,p007871,154.89,1,1,2014-06,train
5,d0000005,80,29.30,1,p219680,0.00,0,0,2006-09,train


In [4]:
# The sensitivity that decision (a) turns on: how much would the positive rate move if same-month
# repeats counted? If this gap is large, the choice is load-bearing and belongs in the presentation.
loose, loose_diag = labels.build_cohorts(donations, same_month_counts=True, verbose=False)
print(f"Positive rate, same-month repeats EXCLUDED (our choice): {diagnostics['positive_rate']:.4f}")
print(f"Positive rate, same-month repeats INCLUDED            : {loose_diag['positive_rate']:.4f}")
print(f"Difference: {abs(loose_diag['positive_rate'] - diagnostics['positive_rate']):.4f}")
print(f"\nShare of donors with >1 gift in their first month: {diagnostics['share_with_multiple_first_month_gifts']:.1%}")

Positive rate, same-month repeats EXCLUDED (our choice): 0.3022
Positive rate, same-month repeats INCLUDED            : 0.4760
Difference: 0.1737

Share of donors with >1 gift in their first month: 21.8%


## 3 · The split — time-based, never random

Train on first-gift cohorts through 2016; hold out 2017 and 2018.

A random split would be indefensible here. Our stakeholder's question is about *next year's* new
donors, so the honest test is whether a model fitted on the past predicts a future it has not seen.
A random split would let it learn from 2018 donors to predict 2017 ones, which is not a situation
she will ever be in, and would flatter the result.

The holdout is not looked at again until a result is final.

In [5]:
by_cohort = (cohorts.groupby("cohort")
             .agg(donors=("donor_id", "size"),
                  repeat_rate=("gave_again", "mean"),
                  median_first_gift=("first_gift_amount", "median"))
             .reset_index())
by_cohort["year"] = by_cohort["cohort"].str[:4]
yearly = by_cohort.groupby("year").agg(donors=("donors", "sum"), repeat_rate=("repeat_rate", "mean"))
print(yearly.to_string(float_format=lambda v: f"{v:,.3f}"))
print("\nIf the repeat rate drifts across years, a model trained on old cohorts will be")
print("miscalibrated on new ones. That is a finding for the error analysis, not a bug.")

      donors  repeat_rate
year                     
2003    2399        0.230
2004    2466        0.230
2005    2430        0.244
2006    2472        0.268
2007    2435        0.264
2008    2396        0.289
2009    2488        0.292
2010    2417        0.298
2011    2431        0.300
2012    2452        0.310
2013    2281        0.333
2014    2370        0.351
2015    2435        0.343
2016    2427        0.340
2017    2450        0.362
2018    2415        0.387

If the repeat rate drifts across years, a model trained on old cohorts will be
miscalibrated on new ones. That is a finding for the error analysis, not a bug.


## 4 · The honest baseline, and why "RFM" is really just "M"

Small nonprofits rank donors by **RFM** — recency, frequency, monetary value. Applied to a cohort of
*first-time* donors it collapses, and the reason is worth saying precisely:

- **Frequency is 1 for everyone.** By construction — they have given exactly once.
- **Recency is identical within a cohort.** Dates are month-level and a cohort *is* a month.
- **Monetary value is the only surviving signal.**

So the strongest honest simple rule is *rank by first gift size*, and that is genuinely what a
development lead does when she builds the list by hand. Albert made this the condition on accepting
a negative result: the baselines have to be **fair**. Beating a strawman would prove nothing.

The scorer measures at her **capacity** — the top slice of each month's new donors she can actually
reach — because that is the decision. Not PR-AUC. The number that matters is dollars of subsequent
giving identified per contact she makes.

One claim we are careful not to make: nobody in this data was randomly assigned to be contacted, so
we cannot measure the *uplift from outreach*. We rank by predicted future value and report value
**identified**, never "retained" or "caused".

In [6]:
holdout = cohorts[cohorts["split"] == "holdout"].reset_index(drop=True)
table = score.compare(holdout, baselines.score_all_baselines(holdout))
table[table["capacity"] == config.STEWARDSHIP_CAPACITY]

,ranking,capacity,donors_contacted,repeat_donors_found,precision,recall,value_identified,value_capture_rate,value_per_contact
2,gift_amount,0.1,499,271,0.543086,0.149065,74951.75,0.432048,150.203908
7,first_month_gift_count,0.1,499,181,0.362725,0.099560,16820.06,0.096957,33.707535
12,random,0.1,499,173,0.346693,0.095160,15085.00,0.086955,30.230461


In [7]:
# Sanity check the scorer itself before trusting any of it: a random ranking at capacity c must
# capture about c of the total value. If that does not hold, the ranking or masking is wrong.
for c in config.CAPACITY_SWEEP:
    row = score.evaluate_at_capacity(holdout, baselines.rank_random(holdout), c)
    print(f"  capacity {c:>5.0%}  →  random captures {row['value_capture_rate']:6.1%} of value  "
          f"(should be ≈ {c:.0%})")

  capacity    1%  →  random captures   0.9% of value  (should be ≈ 1%)
  capacity    5%  →  random captures   3.7% of value  (should be ≈ 5%)
  capacity   10%  →  random captures   8.7% of value  (should be ≈ 10%)
  capacity   20%  →  random captures  20.2% of value  (should be ≈ 20%)
  capacity   50%  →  random captures  50.1% of value  (should be ≈ 50%)


## 5 · Handoff

This notebook produces `data/processed/cohorts.parquet` — one row per donor, labelled and split. It
is the input to all four remaining workstreams:

| Workstream | Owner | Starts from |
|---|---|---|
| Exploratory analysis and features | Reid | `cohorts.parquet` joined back to the projects file |
| Baseline and model development | Rodolfo | `cohorts.parquet`, train split only; score the holdout once |
| Evaluation and error analysis | Thadeus | `evals/score.py`, then calibration and cuts by cohort |
| Outreach economics and recommendations | Malorie | the capacity table above, plus a real cost per contact |

**Open questions this notebook could not settle, for whoever picks them up:**

1. Does the thank-you-packet flag get written *before* or *after* a second gift? If after, it leaks
   the answer and must be dropped as a feature. Albert said to drop that question without regret if
   the timing is ambiguous. Needs the codebook, not the data.
2. Is left-censoring material in the earliest cohorts — are 2003 "first-time" donors genuinely new,
   or did they give before coverage began?
3. Does the projects file let us distinguish a donor's own child's classroom from a stranger's? That
   would be the single strongest feature, and it is not in the donations file.